# VGG-19 learning-rate probe — one cell, old protocol

VGG-19 is the only model where BaCP trails its own I.P. baseline. The question is whether the uniform LR 0.1 explains it.

This runs **one** cell under the protocol that produced the existing 89.99, changing **only the learning rate**, so the two are directly comparable with nothing else moving:

| knob | value | why |
|---|---|---|
| `learning_rate` | **0.05** | the original per-model value; the only change |
| `epochs` / `epochs_ft` | 60 / 50 | as the 89.99 run |
| `delta_T` | 88 | as the 89.99 run |
| `val_split` | **0.0** | as the 89.99 run — the contrastive recipe took no split then, so it trained on all 50,000 |

Reference points, same model / criterion / sparsity:

| | acc |
|---|---|
| dense | 91.42 |
| I.P. magnitude 0.95 | 91.02 |
| BaCP magnitude 0.95, **LR 0.10** | **89.99** |
| BaCP magnitude 0.95, **LR 0.05** | ← this run |

**Reading it.** VGG-19 carries *zero* BatchNorm layers (ResNet-50 has 53), and 85.3% of its weights sit in the classifier MLP that the features feeding the contrastive head pass through. A jump toward 91 means the learning rate dominates and is fixable by restoring the per-model value. Staying near 90 means the cause is architectural — the contrastive signal is computed downstream of the weights the pruner destroys first — and no learning rate will repair it.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Sanity check

In [ ]:
SEED, GPU = 1, 0

# Exactly the 89.99 configuration, with learning_rate as the single change.
OLD_PROTOCOL = dict(epochs=60, epochs_ft=50, delta_T=88, val_split=0.0)

cell = nb.make_cell('vgg19', 'bacp', seed=SEED, pruner='magnitude',
                    sparsity=0.95, variant='oldproto-lr0.05',
                    learning_rate=0.05, **OLD_PROTOCOL)

cfg = cell['config']
print('key   ', cell['key'])
print('config', {k: cfg[k] for k in
                 ('learning_rate', 'epochs', 'epochs_ft', 'delta_T',
                  'val_split', 'contrastive_mode', 'proj_mode', 'tau')})
assert nb.sanity_check([cell]), 'sanity check failed'

## Run — ~25 min

In [ ]:
nb.run(cell, gpu=GPU)
nb.update_results_csv()

## Verdict

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
got = None
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    if r.get('experiment_group') == cell['key'] and r.get('status') == 'ok':
        got = r.get('test_acc_pct')

print(f'{"vgg19 magnitude 0.95":<34}{"acc":>8}')
print(f'{"  dense":<34}{91.42:>8.2f}')
print(f'{"  I.P.":<34}{91.02:>8.2f}')
print(f'{"  BaCP LR 0.10":<34}{89.99:>8.2f}')
print(f'{"  BaCP LR 0.05 (this run)":<34}' +
      (f'{got:>8.2f}' if got is not None else f'{"pending":>8}'))
if got is not None:
    print()
    print(f'vs LR 0.10 : {got - 89.99:+.2f}')
    print(f'vs I.P.    : {got - 91.02:+.2f}')
    print()
    if got - 89.99 > 0.5:
        print('-> learning rate dominates; restore the per-model LR for VGG')
    else:
        print('-> not the learning rate; the cause is architectural')